# Da foto ao gerador de imagens

Exercício complementar de 30 minutos.

**O caso:** o Banco Aurora, uma empresa fictícia, quer usar um cachorro como mascote nas campanhas. Temos só 5 fotos dele.

**Parte 1, entender:** um Vision Transformer lê a foto como uma frase de tokens.

**Parte 2, sonhar:** o Deep Dream inverte a rede e mostra o que ela procura em uma imagem.

**Parte 3, miniprojeto:** você treina um gerador de imagens a partir de texto para desenhar esse mascote.

Antes de começar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

In [ ]:
import gc, os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
RAPIDO = os.environ.get("EX_RAPIDO") == "1"   # teste automatizado, com menos passos

if IN_COLAB:
    !pip install -q -U diffusers peft
    !pip uninstall -q -y torchao

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import snapshot_download
from PIL import Image, ImageOps

transformers.logging.set_verbosity_error()
DEVICE = os.environ.get("EX_DEVICE") or ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(295)
print("Device:", DEVICE)

## 1. As fotos do mascote

As mesmas 5 fotos serão usadas nas três partes.

In [ ]:
PASTA_FOTOS = Path("fotos")
USAR_EXEMPLO = True
if USAR_EXEMPLO:
    snapshot_download("diffusers/dog-example", repo_type="dataset", local_dir=PASTA_FOTOS, allow_patterns="*.jpeg")

def abrir(caminho):
    foto = ImageOps.exif_transpose(Image.open(caminho)).convert("RGB")
    foto.thumbnail((768, 768))
    return foto

fotos = [abrir(p) for p in sorted(PASTA_FOTOS.iterdir()) if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]

def mostrar(imagens, titulos=None, lado=3):
    fig, eixos = plt.subplots(1, len(imagens), figsize=(lado * len(imagens), lado))
    for i, (eixo, img) in enumerate(zip(np.atleast_1d(eixos), imagens)):
        eixo.imshow(img); eixo.axis("off")
        if titulos:
            eixo.set_title(titulos[i], fontsize=9)
    plt.tight_layout(); plt.show()

print(len(fotos), "fotos")
mostrar(fotos)

---
# Parte 1: entender

## 2. A imagem vira tokens

O Vision Transformer corta a imagem em quadrados de 16 × 16 pixels. Uma imagem de 224 × 224 vira 14 × 14 = **196 tokens**.
A partir daí, o funcionamento é o de um encoder como o BERT.

In [ ]:
foto = ImageOps.fit(fotos[0], (224, 224))
P = 16
pedacos = np.array(foto).reshape(14, P, 14, P, 3).swapaxes(1, 2).reshape(-1, P, P, 3)
com_borda = [np.pad(p, ((2, 2), (2, 2), (0, 0)), constant_values=255) for p in pedacos[:42]]
frase = np.concatenate([np.concatenate(com_borda[i:i + 14], axis=1) for i in range(0, 42, 14)], axis=0)

fig, eixos = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={"width_ratios": [1, 2]})
eixos[0].imshow(foto); eixos[0].set_title("Imagem 224 × 224"); eixos[0].axis("off")
eixos[1].imshow(frase); eixos[1].set_title("Os 42 primeiros tokens de 196"); eixos[1].axis("off")
plt.tight_layout(); plt.show()

## 3. O que o modelo vê e para onde olha

O `vit-base-patch16-224` foi treinado em 1.000 categorias. Além da resposta, pedimos a atenção da última camada: quais tokens mais pesaram?
É o mesmo mapa de atenção do mini-GPT, agora sobre uma imagem.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

NOME_VIT = "google/vit-base-patch16-224"
processador = AutoImageProcessor.from_pretrained(NOME_VIT)
vit = AutoModelForImageClassification.from_pretrained(NOME_VIT, attn_implementation="eager").eval().to(DEVICE)
print(f"Parâmetros: {vit.num_parameters():,}")

def olhar(imagem):
    entrada = processador(imagem, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        saida = vit(**entrada, output_attentions=True)
    top = saida.logits[0].softmax(-1).topk(3)
    mapa = saida.attentions[-1][0].mean(0)[0, 1:].reshape(14, 14).cpu().numpy()
    quadrada = ImageOps.fit(imagem, (224, 224))
    fig, eixos = plt.subplots(1, 2, figsize=(7, 3.5))
    eixos[0].imshow(quadrada); eixos[0].axis("off")
    eixos[1].imshow(quadrada); eixos[1].imshow(mapa, cmap="inferno", alpha=0.6, extent=(0, 224, 224, 0), interpolation="bilinear")
    eixos[1].set_title("Atenção"); eixos[1].axis("off")
    plt.tight_layout(); plt.show()
    for p, i in zip(top.values.tolist(), top.indices.tolist()):
        print(f"{p:.0%}  {vit.config.id2label[i]}")

olhar(fotos[0])

Troque o número para ver outra foto. Ele vai de 0 a 4.

In [ ]:
olhar(fotos[1])

---
# Parte 2: sonhar

## 4. Deep Dream

Uma rede treinada responde "o que tem nesta imagem?". O Deep Dream, do Google, inverte a pergunta: congela a rede e altera a **imagem**
para amplificar o que uma camada já detecta. Camadas iniciais realçam texturas. Camadas profundas fazem surgir formas.

Mudar pixels guiado por uma rede é a ideia que reaparece nos geradores de imagem.

In [ ]:
import torchvision.models as tvm

inception = tvm.inception_v3(weights=tvm.Inception_V3_Weights.DEFAULT).eval().to(DEVICE)
inception.requires_grad_(False)

MEDIA = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
DESVIO = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def para_tensor(img):
    return (torch.tensor(np.array(img), device=DEVICE, dtype=torch.float32).permute(2, 0, 1)[None] / 255 - MEDIA) / DESVIO

def para_imagem(t):
    return Image.fromarray(((t * DESVIO + MEDIA).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).astype("uint8"))

def deep_dream(img, nome_camada, oitavas=(0.5, 0.7, 1.0), passos=10, lr=0.03):
    ativacao = {}
    gancho = getattr(inception, nome_camada).register_forward_hook(lambda m, i, o: ativacao.__setitem__("x", o))
    base = para_tensor(img)
    escalar = lambda t, tam: F.interpolate(t, size=tam, mode="bilinear", align_corners=False)
    x = anterior = None
    for escala in oitavas:
        tamanho = [int(s * escala) for s in base.shape[-2:]]
        original = escalar(base, tamanho)
        x = original.clone() if x is None else escalar(x, tamanho) + original - escalar(anterior, tamanho)
        anterior = original
        x = x.detach().requires_grad_(True)
        for _ in range(passos):
            inception(x)
            ativacao["x"].norm().backward()
            with torch.no_grad():
                x += lr * x.grad / (x.grad.abs().mean() + 1e-8)
                x.grad.zero_()
    gancho.remove()
    return para_imagem(x.detach())

LADO = 224 if RAPIDO else 448
PASSOS_SONHO = 2 if RAPIDO else 10
original = ImageOps.fit(fotos[0], (LADO, LADO))
camadas = ["Mixed_5b", "Mixed_6c"]
mostrar([original] + [deep_dream(original, c, passos=PASSOS_SONHO) for c in camadas], ["original"] + camadas, lado=4)

Experimente outras camadas: `Mixed_5d`, `Mixed_6a`, `Mixed_6e` e `Mixed_7a`.

---
# Parte 3: miniprojeto

## 5. O gerador antes do treino

O Stable Diffusion aprendeu a **remover ruído** guiado por um texto. Para gerar, ele parte de ruído puro e limpa passo a passo.

Vamos ensinar a ele uma palavra nova, `sks`, que passa a significar **este** cachorro. Antes do treino, `sks` não quer dizer nada.

In [ ]:
for modelo_antigo in ("vit", "inception"):
    globals().pop(modelo_antigo, None)
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None

from diffusers import DDPMScheduler, StableDiffusionPipeline

BASE = "hf-internal-testing/tiny-stable-diffusion-torch" if RAPIDO else "stable-diffusion-v1-5/stable-diffusion-v1-5"
TAMANHO = 64 if RAPIDO else 512
CLASSE = "dog"
INSTANCIA = f"a photo of sks {CLASSE}"
PROMPTS = [f"a photo of sks {CLASSE} in a bank lobby",
           f"a photo of sks {CLASSE} on the beach",
           f"a watercolor painting of sks {CLASSE}"]

pipe = StableDiffusionPipeline.from_pretrained(
    BASE, torch_dtype=torch.float32, safety_checker=None, requires_safety_checker=False,
    **({} if RAPIDO else {"variant": "fp16"})).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

def misto():
    return torch.autocast("cuda" if DEVICE == "cuda" else "cpu", dtype=torch.float16, enabled=DEVICE == "cuda")

def gerar(prompts, semente=0):
    with misto(), torch.no_grad():
        return [pipe(p, height=TAMANHO, width=TAMANHO, num_inference_steps=2 if RAPIDO else 25, guidance_scale=7.5,
                     generator=torch.Generator("cpu").manual_seed(semente)).images[0] for p in prompts]

inicio = time.time()
antes = gerar(PROMPTS)
print(f"{(time.time() - inicio) / len(PROMPTS):.0f} s por imagem")
mostrar(antes, PROMPTS, lado=3.5)

## 6. Treinar uma LoRA

É a mesma técnica do ajuste fino do mini-LLM. Congelamos o modelo e treinamos só poucos pesos novos nas camadas de atenção.

Cada passo faz sempre a mesma coisa: pega uma foto, adiciona ruído e pede ao modelo para adivinhar esse ruído. O texto dado é `a photo of sks dog`.
Com isso, ele liga a palavra `sks` a este cachorro. As fotos são pré-processadas uma vez, o que acelera o treino.

In [ ]:
from peft import LoraConfig
from torchvision import transforms as T

PASSOS = 3 if RAPIDO else 400
LR = 1e-4
RANK = 8

aumento = T.Compose([T.Resize(TAMANHO), T.RandomCrop(TAMANHO), T.RandomHorizontalFlip(),
                     T.ToTensor(), T.Normalize([0.5] * 3, [0.5] * 3)])
with torch.no_grad():
    lotes = [pipe.vae.encode(aumento(f)[None].to(DEVICE)).latent_dist.sample() for f in fotos for _ in range(10)]
    latentes = torch.cat(lotes) * pipe.vae.config.scaling_factor
    ids = pipe.tokenizer(INSTANCIA, padding="max_length", truncation=True,
                         max_length=pipe.tokenizer.model_max_length, return_tensors="pt").input_ids.to(DEVICE)
    texto = pipe.text_encoder(ids)[0]

pipe.unet.requires_grad_(False)
pipe.unet.add_adapter(LoraConfig(r=RANK, lora_alpha=RANK, init_lora_weights="gaussian",
                                 target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
treinaveis = [p for p in pipe.unet.parameters() if p.requires_grad]
total = sum(p.numel() for p in pipe.unet.parameters())
print(f"Treinando {sum(p.numel() for p in treinaveis):,} de {total:,} pesos "
      f"({sum(p.numel() for p in treinaveis) / total:.1%})")

In [ ]:
agendador = DDPMScheduler.from_config(pipe.scheduler.config)
otimizador = torch.optim.AdamW(treinaveis, lr=LR)
escala = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

pipe.unet.train()
inicio = time.time()
historico = []
for passo in range(1, PASSOS + 1):
    z = latentes[torch.randint(len(latentes), (1,))]
    ruido = torch.randn_like(z)
    t = torch.randint(0, agendador.config.num_train_timesteps, (1,), device=DEVICE)
    with misto():
        previsto = pipe.unet(agendador.add_noise(z, ruido, t), t, texto).sample
        perda = F.mse_loss(previsto.float(), ruido.float())
    otimizador.zero_grad()
    escala.scale(perda).backward()
    escala.step(otimizador)
    escala.update()
    historico.append(perda.item())
    if passo % 50 == 0 or passo == PASSOS:
        print(f"passo {passo}/{PASSOS} | erro médio {np.mean(historico[-50:]):.3f} | {time.time() - inicio:.0f} s", flush=True)
pipe.unet.eval()

## 7. Antes e depois

Mesmos prompts e mesma semente. A única diferença é a LoRA.

In [ ]:
depois = gerar(PROMPTS)
mostrar(antes, [f"ANTES: {p}" for p in PROMPTS], lado=3.5)
mostrar(depois, [f"DEPOIS: {p}" for p in PROMPTS], lado=3.5)

Escreva suas próprias campanhas. Mantenha `sks dog` no prompt e use inglês, porque o modelo foi treinado nesse idioma.

In [ ]:
minhas = ["a photo of sks dog wearing a blue scarf, advertising photography",
          "a cartoon illustration of sks dog holding a credit card"]
mostrar(gerar(minhas, semente=1), minhas, lado=3.5)

## Para responder

1. O mascote gerado se parece com as fotos? Em qual prompt ele mais perde a semelhança?
2. O Vision Transformer e o gerador usam atenção. O que cada um faz com ela?
3. Quais cuidados o Banco Aurora precisa ter antes de usar isso em uma campanha real? Pense em licença, direito de imagem e revisão humana.

## Para ir além, em casa

Repita o miniprojeto com o seu objeto. Em um ambiente novo, crie a pasta `fotos` no painel de arquivos e envie de 5 a 20 fotos variadas.
Depois, mude `USAR_EXEMPLO` para `False`, troque `CLASSE` e `minhas`, e rode tudo.
Use só fotos que você tem direito de usar, e nunca fotos de pessoas sem autorização.